<a href="https://colab.research.google.com/github/Mogi21/ml-algorithms-from-scratch/blob/main/01_ensemble_methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from collections import Counter
from sklearn.tree import DecisionTreeClassifier

class RandomForestFromScratch:
    """
    A custom implementation of a Random Forest classifier.
    Builds an ensemble of decision trees using bootstrap sampling and random subspace feature selection.
    """
    def __init__(self, n_estimators=10, max_depth=None, min_samples_split=2, max_features="sqrt", seed=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.seed = seed
        self.trees = []

    def _bootstrap_sample(self, X, y):
        """Generates a bootstrap sample of the input dataset with replacement."""
        n_samples = X.shape[0]
        indices = np.random.choice(n_samples, n_samples, replace=True)
        return X[indices], y[indices], indices

    def _select_random_features(self, X, n_total_features):
        """Selects a random subset of features to use for a decision tree."""
        if self.max_features == "sqrt":
            max_features = int(np.sqrt(n_total_features))
        elif self.max_features == "log2":
            max_features = int(np.log2(n_total_features))
        else:
            max_features = min(self.max_features, n_total_features)

        feature_indices = np.random.choice(n_total_features, max_features, replace=False)
        return X[:, feature_indices], feature_indices

    def fit(self, X, y):
        """Trains the Random Forest model."""
        self.trees = []
        n_features = X.shape[1]

        if self.seed is not None:
            np.random.seed(self.seed)

        for _ in range(self.n_estimators):
            X_sample, y_sample, _ = self._bootstrap_sample(X, y)
            X_subset, feature_indices = self._select_random_features(X_sample, n_features)

            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                random_state=self.seed
            )
            tree.fit(X_subset, y_sample)
            self.trees.append((tree, feature_indices))

    def predict(self, X):
        """Predicts class labels using majority voting."""
        predictions = np.zeros((X.shape[0], len(self.trees)))
        for i, (tree, feature_indices) in enumerate(self.trees):
            predictions[:, i] = tree.predict(X[:, feature_indices])

        # Apply majority voting
        return np.apply_along_axis(lambda x: Counter(x).most_common(1)[0][0], axis=1, arr=predictions)

In [2]:
class AdaBoostFromScratch:
    """
    Adaptive Boosting (AdaBoost) classifier for binary classification.
    Iteratively adjusts sample weights to focus on misclassified samples.
    """
    def __init__(self, n_estimators=50, seed=None):
        self.n_estimators = n_estimators
        self.seed = seed
        self.alphas = []
        self.models = []

    def _compute_error(self, y, y_pred, w):
        """Calculates the weighted training error."""
        return np.sum(w[y != y_pred])

    def _compute_alpha(self, error):
        """Computes the weight (alpha) of the current weak learner."""
        epsilon = 1e-10  # Prevent division by zero
        return 0.5 * np.log((1.0 - error + epsilon) / (error + epsilon))

    def _update_weights(self, w, alpha, y, y_pred):
        """Updates and normalizes sample weights."""
        w = w * np.exp(-alpha * y * y_pred)
        return w / np.sum(w)

    def fit(self, X, y):
        """Trains the AdaBoost ensemble."""
        n_samples, n_features = X.shape
        self.models = []
        self.alphas = []

        # Ensure labels are exactly {-1, 1}
        if set(np.unique(y)) == {0, 1}:
            y = np.where(y == 0, -1, 1)

        # Initialize uniform weights
        w = np.full(n_samples, 1.0 / n_samples)

        if self.seed is not None:
            np.random.seed(self.seed)

        for _ in range(self.n_estimators):
            stump = DecisionTreeClassifier(max_depth=1, random_state=self.seed)
            stump.fit(X, y, sample_weight=w)
            y_pred = stump.predict(X)

            error = self._compute_error(y, y_pred, w)
            alpha = self._compute_alpha(error)
            w = self._update_weights(w, alpha, y, y_pred)

            self.models.append(stump)
            self.alphas.append(alpha)

    def predict(self, X):
        """Predicts class labels via weighted sum of weak learners."""
        weak_preds = np.array([stump.predict(X) for stump in self.models])
        weighted_sum = np.dot(self.alphas, weak_preds)

        final_preds = np.sign(weighted_sum)
        final_preds[final_preds == 1] = 1
        final_preds[final_preds == -1] = -1

        return final_preds

In [3]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

print("Loading Breast Cancer Dataset...")
data = load_breast_cancer()
X = data.data
y_rf = data.target
# AdaBoost requires {-1, 1} labels
y_ada = np.where(data.target == 0, -1, 1)

# --- Random Forest Evaluation ---
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(X, y_rf, test_size=0.2, random_state=42)
rf = RandomForestFromScratch(n_estimators=10, max_depth=5, max_features="sqrt", seed=42)
rf.fit(X_train_rf, y_train_rf)
rf_pred = rf.predict(X_test_rf)
print(f"Random Forest Accuracy: {accuracy_score(y_test_rf, rf_pred):.4f}")

# --- AdaBoost Evaluation ---
X_train_ada, X_test_ada, y_train_ada, y_test_ada = train_test_split(X, y_ada, test_size=0.2, random_state=42)
ada = AdaBoostFromScratch(n_estimators=50, seed=42)
ada.fit(X_train_ada, y_train_ada)
ada_pred = ada.predict(X_test_ada)
print(f"AdaBoost Accuracy:      {accuracy_score(y_test_ada, ada_pred):.4f}")

Loading Breast Cancer Dataset...
Random Forest Accuracy: 0.9649
AdaBoost Accuracy:      0.9649
